# 1. Preparation   
This is a Python Notebook.  
To make it work, you need to press "play" on the code cells.   
Remember to always run cells in the right order and never skip one!   
In case of doubt, you can always restart from the beginning.   

First, let's clone the course repository so we can access our curated dataset of annotated passages (YES / NO).

In [ ]:
# Clone the course repository to access the dataset
!git clone https://github.com/SimoneRebora/CMCLS

In [ ]:
import pandas as pd

# 1. Read the first curated dataset (100 passages)
df_1 = pd.read_csv(
    '/content/CMCLS/materials/HoD_annotated_dataset.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['Sentences', 'curation']]

# 2. Read the class curated dataset (60 passages) and align column name
df_2 = pd.read_csv(
    '/content/CMCLS/materials/Hands_on_5_curation_first_round.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['text', 'curation']].rename(columns={'text': 'Sentences'})

# 3. Merge into 160 passages
my_df = pd.concat([df_1, df_2], ignore_index=True)

# 4. Remove empty rows and standardize text/labels
my_df = my_df.dropna(subset=['Sentences', 'curation']).copy()
my_df['Sentences'] = my_df['Sentences'].astype(str).str.strip()
my_df['curation'] = my_df['curation'].astype(str).str.strip().str.upper()

# Show the dataset table on screen
my_df

# Save the merged curated dataset as a CSV file
#my_df.to_csv('merged_curated_dataset.csv', index=False)

2. Machine Learning:

Before training the computer, we divide our dataset into two parts:

Train (80%): The study material the computer learns from (with the answers).

Test (20%): The hidden final exam we use to check if the computer actually learned.

In [ ]:
# Import the function that splits our table into train and test sets
from sklearn.model_selection import train_test_split

# Split 'my_df' into 80% training data and 20% testing data
# random_state=123 ensures the random shuffle is repeatable
train, test = train_test_split(my_df, test_size=0.2, random_state=123)

# Print how many passages are in each pile
print("Training passages:", len(train))
print("Testing passages:", len(test))

#Create word frequency features

We use CountVectorizer to turn each passage into a matrix counting how many times each vocabulary word appears.

In [ ]:
# Import CountVectorizer to convert text into word frequency counts
from sklearn.feature_extraction.text import CountVectorizer

# Set up the word counter:
# - lowercase=True: treats "Other" and "other" as the same word
# - stop_words='english': ignores common filler words like "the", "and", "is"
# - min_df=2: ignores ultra-rare words that appear in fewer than 2 passages
vectorizer = CountVectorizer(lowercase=True, stop_words='english', min_df=2)

# Learn the vocabulary from the training passages and count word frequencies (train_X)
train_X = vectorizer.fit_transform(train['Sentences'])
# Save the true YES/NO answers for the training passages (train_y)
train_y = train['curation']

# Count the same vocabulary words in the hidden test passages (test_X)
# Note: We use .transform() (not .fit_transform()) so the model doesn't peek at the exam!
test_X = vectorizer.transform(test['Sentences'])
# Save the true YES/NO answers for the test passages (test_y)
test_y = test['curation']

# Print the dimensions of our number tables: (number of passages, number of unique words)
print("Train table shape:", train_X.shape)
print("Test table shape:", test_X.shape)

# Display the word-frequency table used for training
train_features = pd.DataFrame(
    train_X.toarray(),
    columns=vectorizer.get_feature_names_out()
)
train_features.insert(0, 'Sentence', train['Sentences'].values)
train_features

# Save the word-frequency ("embeddings") table as a CSV file
#train_features.to_csv('word_frequency_features.csv', index=False)

#Logistic regression
Train a logistic regression model with word frequencies (to predict YES vs. NO).

In [ ]:
# Import the Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Create an empty Logistic Regression model
# max_iter=1000 gives the math algorithm plenty of steps to find the best word weights
model = LogisticRegression(max_iter=1000)

# Train ("fit") the model by showing it the training word counts (train_X) and answers (train_y)
model.fit(train_X, train_y)

#Make predictions with the model on the test set
Now we hide the true answers on the 20% test set and ask our trained model to guess YES or NO for each passage.

In [ ]:
# Ask the trained model to predict YES or NO for the test passages
prediction = model.predict(test_X)

# Add the model's guesses as a new column in our 'test' table so we can compare side-by-side
test['logistic regression'] = prediction

# Display the test table with true labels and model predictions
test

# Save the test predictions table as a CSV file
# test.to_csv('test_predictions.csv', index=False)


#See the efficiency scores
Let's print the official report card comparing the human annotations (test_y) against the model's guesses (prediction).

In [ ]:
# Import the classification report tool to grade the model's predictions
from sklearn.metrics import classification_report

# Print precision, recall, and f1-score comparing true answers vs. model predictions
print(classification_report(test_y, prediction))

# Save the classification report as a CSV file
#report_df = pd.DataFrame(classification_report(test_y, prediction, output_dict=True)).transpose()
#report_df.to_csv('classification_report.csv')

#Which words mattered most?
The top 10 words that pushed the model to guess YES versus NO.

In [ ]:
# Import numpy to help sort the word weights
import numpy as np

# Get the list of all vocabulary words
words = np.array(vectorizer.get_feature_names_out())
# Get the importance weight ("clue score") the model assigned to each word
coefs = model.coef_[0]

# Put the top 10 YES words and top 10 NO words into a clean table
top_yes_idx = np.argsort(coefs)[-10:][::-1]
top_no_idx = np.argsort(coefs)[:10]

top_words_df = pd.DataFrame({
    'Top_YES_word': words[top_yes_idx],
    'YES_score': np.round(coefs[top_yes_idx], 3),
    'Top_NO_word': words[top_no_idx],
    'NO_score': np.round(coefs[top_no_idx], 3)
})

top_words_df

# 3. Your turn: Save your results

Choose **one** table to save for Moodle by removing the `#` before its its saving code (last line/s of the code) and rerunning that cell:

**Word-frequency table ("embeddings"):** Cell 6 (`# train_features.to_csv(...)`)

**Test predictions:** Cell 10 (`# test.to_csv(...)`)

**Efficiency scores:** Cell 12 (`# report_df = ...` and `# report_df.to_csv(...)`)

Then open the **Files** folder on the left sidebar, click the three dots next to your `.csv` file, and select **Download**.